# Day 2 — automatic backward via topo-sort

Yesterday you wrote `_backward` closures by hand for one specific graph and called them in the right order manually. Today we generalize: **any** graph, **one** `.backward()` call.

## What you ship today

1. `Value.backward()` that walks the graph in reverse topological order and calls each node's `_backward` closure.
2. New operators: `exp`, `**` (scalar power), `/`, `relu`, `__neg__`, `__sub__`, `__radd__`, `__rmul__`.
3. Gradient-check 5 random expressions against PyTorch — all should agree to atol=1e-5.

## The single most important idea

The `_backward` for each op is **just the local derivative**, multiplied by the *output* node's already-accumulated `.grad`. The chain rule is the multiplication; the topological order ensures every node's `.grad` is fully accumulated before that node's `_backward` runs.

## Traps to avoid (from README.md)

1. **Accumulate with `+=`, never assign.** A Value can be reused (e.g. `b = a + a`); each use contributes a separate gradient term.
2. **Topo order is post-order DFS, then reversed.** Pre-order will mostly work and silently break on diamonds like `b = a + a; c = b * b`.
3. **`__radd__` / `__rmul__`** exist for `2 + v` / `2 * v`. One-liner each, easy to forget.

In [ ]:
import math
import random
import torch
from typing import Callable

print(f'torch {torch.__version__}  mps={torch.backends.mps.is_available()}')

## The full `Value` class

Same skeleton as Day 1 (`data`, `grad`, `_prev`, `_op`, `_backward`), but now with:

- A real `backward()` that does topo-sort + reverse traversal
- Six operations, each with the correct local derivative in `_backward`
- Right-hand operators so plain Python numbers can sit on either side of `+`, `*`, `/`, `-`

Read each operation's `_backward` and notice **the same pattern every time**: `self.grad += (local derivative) * out.grad`. The `out.grad` factor is the chain rule. Everything else is one line of calculus.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None  # leaves do nothing on backward

    def __repr__(self):
        return f'Value(data={self.data:.4g}, grad={self.grad:.4g})'

    # ----- operations -----
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), 'only scalar exponents supported'
        out = Value(self.data ** exponent, (self,), f'**{exponent}')
        def _backward():
            self.grad += (exponent * self.data ** (exponent - 1)) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            # d(e^x)/dx = e^x — and that's exactly out.data
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), 'relu')
        def _backward():
            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    # ----- right-hand / negation / sub / div (glue) -----
    def __neg__(self):                              # -self
        return self * -1

    def __radd__(self, other):                       # other + self
        return self + other

    def __sub__(self, other):                        # self - other
        return self + (-other if isinstance(other, Value) else Value(-other))

    def __rsub__(self, other):                       # other - self
        return Value(other) - self

    def __rmul__(self, other):                       # other * self
        return self * other

    def __truediv__(self, other):                    # self / other
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1

    def __rtruediv__(self, other):                   # other / self
        return Value(other) * self ** -1

    # ----- the only "smart" method -----
    def backward(self):
        # 1. topological order: post-order DFS, leaves first, root last
        topo, visited = [], set()
        def build(v):
            if v in visited: return
            visited.add(v)
            for parent in v._prev:
                build(parent)
            topo.append(v)
        build(self)

        # 2. seed: d(self)/d(self) = 1
        self.grad = 1.0

        # 3. walk in reverse: each node's _backward pushes gradient to its parents
        for v in reversed(topo):
            v._backward()

print('Value class defined.')

## Test 1: the diamond trap

If you use a Value twice — `b = a + a` — and `_backward` did `=` instead of `+=`, gradient on `a` would be computed once and overwritten by the second pass, losing the contribution.

Here we compute `c = (a + a) * (a + a) = (2a)² = 4a²`. By calculus `dc/da = 8a`. At `a=3`, the expected gradient is `24`.

If you see `12`, you've got the `=`-vs-`+=` bug — gradient was overwritten.
If you see `24`, accumulation is correct.

In [ ]:
a = Value(3.0)
b = a + a       # b = 2a
c = b * b       # c = (2a)^2 = 4a^2
c.backward()
print(f'c.data = {c.data}  (expected 36 = 4*9)')
print(f'a.grad = {a.grad}  (expected 24 = 8*3)')
assert abs(a.grad - 24.0) < 1e-9, 'diamond test failed — check += in _backward'

## Test 2: gradient check against PyTorch — 5 random-ish expressions

These cover every operation we added. Each one builds the *same* expression twice — once with `Value`, once with `torch.tensor` — runs `.backward()` on both, and asserts the resulting gradients match to `atol=1e-5`.

If any of these print `MISMATCH`, the bug is in that operation's `_backward`.

In [ ]:
def grad_check(name, f_micro, f_torch, a_val=2.0, b_val=-3.0, c_val=0.5, atol=1e-5):
    # micrograd side
    a = Value(a_val); b = Value(b_val); c = Value(c_val)
    out = f_micro(a, b, c)
    out.backward()
    m_grads = (a.grad, b.grad, c.grad)

    # PyTorch side — use float64 so the comparison isn't bottlenecked by float32 noise
    ta = torch.tensor(a_val, requires_grad=True, dtype=torch.float64)
    tb = torch.tensor(b_val, requires_grad=True, dtype=torch.float64)
    tc = torch.tensor(c_val, requires_grad=True, dtype=torch.float64)
    t_out = f_torch(ta, tb, tc)
    t_out.backward()
    t_grads = (ta.grad.item(), tb.grad.item(), tc.grad.item())

    ok = all(abs(m - t) < atol for m, t in zip(m_grads, t_grads))
    print(f'{name:24s}  micrograd={m_grads}  torch={t_grads}  {"ok" if ok else "MISMATCH"}')
    assert ok, f'{name} failed'

# Five expressions, one per operation family.
grad_check('add_mul',     lambda a,b,c: (a + b) * c,                         lambda a,b,c: (a + b) * c)
grad_check('tanh_chain',  lambda a,b,c: (a * b + c).tanh(),                  lambda a,b,c: (a * b + c).tanh())
grad_check('relu_pow',    lambda a,b,c: (a * b - c).relu() + a ** 2,         lambda a,b,c: (a * b - c).relu() + a ** 2)
grad_check('exp_div',     lambda a,b,c: (a.exp() + b) / (c + 1),             lambda a,b,c: (a.exp() + b) / (c + 1))
grad_check('mixed',       lambda a,b,c: ((a + 2) * (b - c)).tanh() + a / b,  lambda a,b,c: ((a + 2) * (b - c)).tanh() + a / b)

print('\nAll 5 expressions agree with PyTorch to atol=1e-5.')

## End-of-day check

- [x] `Value.backward()` works on any graph — diamond test passes
- [x] All 5 random gradient-check expressions agree with PyTorch to atol=1e-5
- [ ] **You can articulate why the topo order needs to be reversed before the backward sweep.**

If the last box doesn't feel sharp, write the answer to yourself before moving on.

Day 3 (notebook `03_neuron_layer_mlp.ipynb`) builds `Neuron` → `Layer` → `MLP` on top of this `Value`. Same engine, just stacked.

Append one line to `NOTES.md`: what was the most surprising thing about today?